In [38]:
import os
import pickle
import json
from tqdm import tqdm
import numpy as np
import re

#parameters
KNN = 8
REASONING = False

#paths
g4_path = os.path.join("2-Build_Graph/data/g4.pkl")
knn_path = os.path.join("Evaluation", "data", "knn.jsonl")
context_path = os.path.join("Evaluation", "data", f"context_{KNN}nn{"_reasoning" if REASONING else ""}_reranked.jsonl")
question_path = os.path.normpath(os.path.join( "InfoSeek", "sampled_questions.jsonl"))
evaluated_base_path = os.path.join("Evaluation", "data", f"evaluation_{KNN}nn{"_reasoning" if REASONING else ""}_base.jsonl")
evaluated_path = os.path.join("Evaluation", "data", f"evaluation_{KNN}nn{"_reasoning" if REASONING else ""}.jsonl")
evaluated_sub_path = os.path.join("EvaluationSub", "data", f"evaluation_{KNN}nn{"_reasoning" if REASONING else ""}.jsonl")
eval_base_path = os.path.join("Evaluation", "data", f"evaluation_context_recall_{KNN}nn{"_reasoning" if REASONING else ""}_base.jsonl")
eval_path = os.path.join("Evaluation", "data", f"evaluation_context_recall_{KNN}nn{"_reasoning" if REASONING else ""}.jsonl")
eval_sub_path = os.path.join("EvaluationSub", "data", f"evaluation_context_recall_{KNN}nn{"_reasoning" if REASONING else ""}.jsonl")

In [39]:
#load data
with open(g4_path, "rb") as f:
    nodes = pickle.load(f)

questions = {}
with open(question_path, "r", encoding="utf-8") as f:
    for line in f:
        line = json.loads(line)
        answer_eval = line["answer_eval"]
        if not isinstance(answer_eval, list):
            print(f"Invalid answer_eval for question {line['data_id']}: {answer_eval}")
            raise ValueError("Invalid answer_eval format")
        for i in range(len(answer_eval)):
            val = answer_eval[i]
            if not (isinstance(val, (str, int, float)) or (isinstance(val, dict) and "wikidata" in val and "range" in val)):
                print(f"Invalid answer_eval value for question {line['data_id']}: {val}")
                raise ValueError("Invalid answer_eval value format")
            if isinstance(val, dict):
                answer_eval[i]["value"] = answer_eval[i].pop("wikidata")
        questions[line["data_id"]] = (line["question"], line["answer_eval"])

In [40]:
knns = {}
with open(knn_path, "r", encoding="utf-8") as f:
    for line in f:
        line = json.loads(line)
        knns[line["qid"]] = line["knn"][:KNN]

contexts_base = {}
with open(context_path, "r", encoding="utf-8") as f:
    for line in f:
        line = json.loads(line)
        current_context = [nid for nid in line["sorted_context_nodes"] if nid in knns[line["qid"]]]
        if len(current_context) != KNN:
            print(f"Warning: Question {line['qid']} has {len(current_context)} context nodes instead of {KNN}")
        contexts_base[line["qid"]] = current_context

contexts = {}
with open(context_path, "r", encoding="utf-8") as f:
    for line in f:
        line = json.loads(line)
        contexts[line["qid"]] = line["sorted_context_nodes"]

contexts_sub = {}
for level in range(100):
    context_sub_path = os.path.normpath(os.path.join("EvaluationSub", "data", f"context_{KNN}nn_reranked_{level}.jsonl"))
    if not os.path.exists(context_sub_path):
        break
    with open(context_sub_path, "r", encoding="utf-8") as f:
        for line in f:
            line = json.loads(line)
            contexts_sub[line["qid"]] = line["sorted_context_nodes"]

In [41]:
#load evaluated
evaluated_base = {}
if os.path.exists(evaluated_base_path):
    with open(evaluated_base_path, "r", encoding="utf-8") as f:
        for line in f:
            line = json.loads(line)
            score = int(line["score"])
            if score in {0,1}:
                evaluated_base[line["qid"]] = score

evaluated = {}
if os.path.exists(evaluated_path):
    with open(evaluated_path, "r", encoding="utf-8") as f:
        for line in f:
            line = json.loads(line)
            score = int(line["score"])
            if score in {0,1}:
                evaluated[line["qid"]] = score

evaluated_sub = {}
if os.path.exists(evaluated_sub_path):
    with open(evaluated_sub_path, "r", encoding="utf-8") as f:
        for line in f:
            line = json.loads(line)
            score = int(line["score"])
            if score in {0,1}:
                evaluated_sub[line["qid"]] = score

In [42]:
#load_progress:
processed_questions_base = {}
if os.path.exists(eval_base_path):
    with open(eval_base_path, "r", encoding="utf-8") as f:
        for line in f:
            line = json.loads(line)
            value = line["context_recall"]
            if np.isnan(value) or value == 0:
                continue
            processed_questions_base[line["qid"]] = 1

processed_questions = {}
if os.path.exists(eval_path):
    with open(eval_path, "r", encoding="utf-8") as f:
        for line in f:
            line = json.loads(line)
            value = line["context_recall"]
            if np.isnan(value) or value == 0:
                continue
            processed_questions[line["qid"]] = 1

processed_questions_sub = {}
if os.path.exists(eval_sub_path):
    with open(eval_sub_path, "r", encoding="utf-8") as f:
        for line in f:
            line = json.loads(line)
            value = line["context_recall"]
            if np.isnan(value) or value == 0:
                continue
            processed_questions_sub[line["qid"]] = 1

In [43]:
for qid in tqdm(questions.keys()):
    if qid in processed_questions_base:
        continue
    if qid in evaluated_base and evaluated_base[qid] == 1:
        processed_questions_base[qid] = 1
    else:
        question = questions[qid][0]
        answer_eval = questions[qid][1]

        context_nodes = contexts_base[qid]
        for c in context_nodes:
            if nodes[c].node_type == "V":
                continue
            content = nodes[c].content
            if any(isinstance(ans,str) and re.search(rf"\b{re.escape(ans)}\b", content, re.IGNORECASE) for ans in answer_eval):
                processed_questions_base[qid] = 1

print(sum(processed_questions_base.values()))

100%|██████████| 1000/1000 [00:00<00:00, 6447.35it/s]

791


In [44]:
for qid in tqdm(questions.keys()):
    if qid in processed_questions:
        continue
    if qid in evaluated and evaluated[qid] == 1:
        processed_questions[qid] = 1
    
    elif qid in processed_questions_base and processed_questions_base[qid] == 1:
        processed_questions[qid] = 1
    else:
        question = questions[qid][0]
        answer_eval = questions[qid][1]

        context_nodes = contexts[qid]
        for c in context_nodes:
            if nodes[c].node_type == "V":
                continue
            content = nodes[c].content
            if any(isinstance(ans,str) and re.search(rf"\b{re.escape(ans)}\b", content, re.IGNORECASE) for ans in answer_eval):
                processed_questions[qid] = 1
print(sum(processed_questions.values()))

100%|██████████| 1000/1000 [00:00<00:00, 5460.03it/s]

836


In [45]:
for qid in tqdm(questions.keys()):
    if qid in processed_questions_sub:
        continue
    if qid in evaluated_sub and evaluated_sub[qid] == 1:
        processed_questions_sub[qid] = 1
    else:
        question = questions[qid][0]
        answer_eval = questions[qid][1]

        context_nodes = contexts_sub[qid]
        for c in context_nodes:
            if nodes[c].node_type == "V":
                continue
            content = nodes[c].content
            if any(isinstance(ans,str) and re.search(rf"\b{re.escape(ans)}\b", content, re.IGNORECASE) for ans in answer_eval):
                processed_questions_sub[qid] = 1
print(sum(processed_questions_sub.values()))

100%|██████████| 1000/1000 [00:00<00:00, 3062.85it/s]

803


In [46]:
for qid in questions:
    if processed_questions.get(qid,0) == 0:
        question = questions[qid][0]
        answer_eval = questions[qid][1]
        answer_str = []
        for item in answer_eval:
            if isinstance(item, str):
                answer_str.append(item)
            elif isinstance(item, dict):
                continue
                answer_str.append(f"{item['value']} ({item['range'][0]} to {item['range'][1]})")
        answer_str = "; ".join(answer_str)
        if not answer_str:
            continue
        answer_str = "Acceptable answers are: " + answer_str
        print(qid," - ",question, " - ", answer_str)
        context_nodes = contexts[qid]
        print([nodes[nid].content for nid in context_nodes])
        print("-"*30)

infoseek_train_00040090  -  What is the closest parent taxonomy of this plant?  -  Acceptable answers are: Comfrey; Symphytum
["Verbena hastata, also known as American vervain, blue vervain, or swamp verbena, is a perennial herbaceous flowering plant in the family Verbenaceae. Found across North America, this common, hardy, and drought-resistant plant is characterized by stiffly erect, branching square stems and opposite, simple leaves with double-serrate margins. It produces purple flowers in the summer and its Latin epithet, 'hastata,' means 'spear-shaped.' As a member of the diploid North American vervains, it possesses 14 chromosomes. Hybridization is believed to have played a role in its evolution, potentially involving species such as white vervain (V. urticifolia), V. lasiostachys, V. menthifolia, or V. orcuttiana. Additionally, Verbena hastata serves as a larval host to the common buckeye butterfly and was involved in an incident of chloroplast transfer to the mock vervain Glan